# MRI to PET Benchmark
Models (all base=16, ≈1.5 M params, same lr/epochs/batch):
1. UNet‑L1 (plain UNet, L1 only)
2. UNet‑SSIM+GDL (plain UNet, same triple loss)
3. Pix2Pix3D (GAN: UNet generator + PatchGAN + L1)
4. ResNet3D (residual 3D encoder‑decoder)
5. DenseNet3D (dense‑connected 3D encoder‑decoder)
6. NearestNeighbor (non‑parametric retrieval baseline)

Metrics use **brain‑masked SSIM/MAE/MSE**.


In [1]:
DATA_ROOT = "/workspace/Data/NACC_Combined_AUG"  
CKPT_DIR  = "/workspace/checkpoints/benchmark_v3"  

import os; 
os.makedirs(CKPT_DIR, exist_ok=True)


In [2]:
import os, sys, random, math, glob, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from scipy.ndimage import map_coordinates, gaussian_filter, binary_dilation, binary_erosion
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from skimage.metrics import structural_similarity as sk_ssim

torch.manual_seed(42); np.random.seed(42); random.seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


Device: cuda


## Normalisation, masking, metrics

In [3]:
def norm_mri(v):
    lo, hi = np.percentile(v, 1), np.percentile(v, 99)
    return np.clip((v - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)

def norm_pet(v):
    fg_mask = v > (v.max() * 0.05 + 1e-8)
    mean_uptake = v[fg_mask].mean() if fg_mask.any() else 1.0
    suv = v / (mean_uptake + 1e-8)
    return np.clip(suv / 2.0, 0, 1).astype(np.float32)

def compute_brain_mask(mri_norm, threshold=0.1):
    mask = mri_norm > threshold
    for _ in range(3): mask = binary_dilation(mask)
    for _ in range(3): mask = binary_erosion(mask)
    return mask.astype(bool)

def masked_ssim_slice(gt_slice, pred_slice, mask_slice, data_range=1.0):
    if mask_slice.sum() < 100:
        return float(sk_ssim(gt_slice, pred_slice, data_range=data_range))
    return float(sk_ssim(gt_slice*mask_slice, pred_slice*mask_slice, data_range=data_range))

def compute_masked_metrics(syn, gt, mask):
    mae_g  = float(np.mean(np.abs(syn - gt)))
    mse_g  = float(np.mean((syn - gt)**2))
    ssim_g = float(np.mean([sk_ssim(gt[i], syn[i], data_range=1.0) for i in range(gt.shape[0])]))
    syn_m, gt_m = syn[mask], gt[mask]
    mae_m  = float(np.mean(np.abs(syn_m - gt_m)))
    mse_m  = float(np.mean((syn_m - gt_m)**2))
    ssim_m = float(np.mean([masked_ssim_slice(gt[i], syn[i], mask[i]) for i in range(gt.shape[0])]))
    return {'MAE':mae_g, 'MSE':mse_g, 'SSIM':ssim_g,
            'MAE_brain':mae_m, 'MSE_brain':mse_m, 'SSIM_brain':ssim_m}


## Augmentations and dataset

In [4]:
PATCH_SIZE = 64

class PETDataset(Dataset):
    def __init__(self, pairs, patch_size=PATCH_SIZE):
        self.pairs=pairs; self.patch_size=patch_size
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        mri_path, pet_path = self.pairs[idx]
        mri = norm_mri(np.load(mri_path)); pet = norm_pet(np.load(pet_path))
        D,H,W = mri.shape; p=self.patch_size
        x=random.randint(0,D-p); y=random.randint(0,H-p); z=random.randint(0,W-p)
        mri = mri[x:x+p,y:y+p,z:z+p]; pet = pet[x:x+p,y:y+p,z:z+p]
        return torch.from_numpy(mri.copy()).unsqueeze(0), torch.from_numpy(pet.copy()).unsqueeze(0)


## Data split (patient-level) from DATA_ROOT

In [5]:
patient_pairs = defaultdict(list)
for folder in sorted(os.listdir(DATA_ROOT)):
    fd=os.path.join(DATA_ROOT, folder); mri=os.path.join(fd,"mri_processed.npy"); pet=os.path.join(fd,"pet_processed.npy")
    if not (os.path.isdir(fd) and os.path.isfile(mri) and os.path.isfile(pet)):
        continue
    if folder.endswith('_orig'): pid = folder[:-5]
    elif '_aug_' in folder: pid = folder[:folder.index('_aug_')]
    else: pid = folder
    patient_pairs[pid].append((mri, pet))

patient_ids = sorted(patient_pairs.keys()); random.seed(42); random.shuffle(patient_ids)
n_total=len(patient_ids); n_tr=int(n_total*0.8); n_va=int(n_total*0.1)
tr_ids = patient_ids[:n_tr]; va_ids = patient_ids[n_tr:n_tr+n_va]; te_ids = patient_ids[n_tr+n_va:]
tr_pairs = [p for pid in tr_ids for p in patient_pairs[pid]]
va_pairs = [p for pid in va_ids for p in patient_pairs[pid]]
te_pairs = [p for pid in te_ids for p in patient_pairs[pid]]
print(f"Patients: {n_total}  (train {len(tr_ids)}, val {len(va_ids)}, test {len(te_ids)})")
print(f"Pairs: train {len(tr_pairs)}  val {len(va_pairs)}  test {len(te_pairs)}")

BATCH = 2; 
EPOCHS = 20; 
LR = 2e-4; 
WD = 1e-4; 
USE_TTA=False
train_loader = DataLoader(PETDataset(tr_pairs), batch_size=BATCH, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(PETDataset(va_pairs), batch_size=1, shuffle=False, num_workers=2)
test_loader  = DataLoader(PETDataset(te_pairs), batch_size=1, shuffle=False, num_workers=2)


Patients: 259  (train 207, val 25, test 27)
Pairs: train 1863  val 225  test 243


## Models (param-matched base=16)

In [6]:
def gn(c, groups=8):
    g=min(groups,c);
    while c%g!=0: g//=2
    return nn.GroupNorm(g,c)

def block(ic, oc):
    return nn.Sequential(
        nn.Conv3d(ic, oc, 3, padding=1), gn(oc), nn.LeakyReLU(0.1, inplace=True),
        nn.Conv3d(oc, oc, 3, padding=1), gn(oc), nn.LeakyReLU(0.1, inplace=True),
    )

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__();
        self.W_g = nn.Sequential(nn.Conv3d(F_g, F_int, 1, bias=True), gn(F_int))
        self.W_x = nn.Sequential(nn.Conv3d(F_l, F_int, 1, bias=True), gn(F_int))
        self.psi = nn.Sequential(nn.Conv3d(F_int,1,1,bias=True), nn.Sigmoid())
        self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x):
        g_up = F.interpolate(g, size=x.shape[2:], mode='trilinear', align_corners=False)
        att = self.psi(self.relu(self.W_g(g_up)+self.W_x(x)))
        return x*att

class UNetAtt(nn.Module):
    def __init__(self, base=16):
        super().__init__(); b=base
        self.e1=block(1,b); self.e2=block(b,b*2); self.e3=block(b*2,b*4); self.bn=block(b*4,b*8)
        self.ag3=AttentionGate(b*8,b*4,b*2); self.ag2=AttentionGate(b*4,b*2,b); self.ag1=AttentionGate(b*2,b,b//2)
        self.d3=block(b*8+b*4,b*4); self.d2=block(b*4+b*2,b*2); self.d1=block(b*2+b,b)
        self.out=nn.Conv3d(b,1,1); self.pool=nn.MaxPool3d(2); self.up=nn.Upsample(scale_factor=2, mode='trilinear', align_corners=False)
    def forward(self,x):
        s1=self.e1(x); s2=self.e2(self.pool(s1)); s3=self.e3(self.pool(s2)); btm=self.bn(self.pool(s3))
        s3a=self.ag3(btm,s3); x=self.d3(torch.cat([self.up(btm), s3a],1))
        s2a=self.ag2(x,s2);   x=self.d2(torch.cat([self.up(x), s2a],1))
        s1a=self.ag1(x,s1);   x=self.d1(torch.cat([self.up(x), s1a],1))
        return torch.sigmoid(self.out(x))

def conv_block(ic, oc):
    return nn.Sequential(
        nn.Conv3d(ic, oc, 3, padding=1), nn.InstanceNorm3d(oc), nn.LeakyReLU(0.1, inplace=True),
        nn.Conv3d(oc, oc, 3, padding=1), nn.InstanceNorm3d(oc), nn.LeakyReLU(0.1, inplace=True),
    )

class UNetPlain(nn.Module):
    def __init__(self, base=16):
        super().__init__(); b=base
        self.e1=conv_block(1,b); self.e2=conv_block(b,b*2); self.e3=conv_block(b*2,b*4); self.bn=conv_block(b*4,b*8)
        self.pool=nn.MaxPool3d(2); self.up=nn.Upsample(scale_factor=2, mode='trilinear', align_corners=False)
        self.d3=conv_block(b*8+b*4,b*4); self.d2=conv_block(b*4+b*2,b*2); self.d1=conv_block(b*2+b,b); self.out=nn.Conv3d(b,1,1)
    def forward(self,x):
        s1=self.e1(x); s2=self.e2(self.pool(s1)); s3=self.e3(self.pool(s2)); btm=self.bn(self.pool(s3))
        x=self.d3(torch.cat([self.up(btm), s3],1)); x=self.d2(torch.cat([self.up(x), s2],1)); x=self.d1(torch.cat([self.up(x), s1],1))
        return torch.sigmoid(self.out(x))

class PatchDiscriminator3D(nn.Module):
    def __init__(self, base=16):
        super().__init__(); b=base
        self.net=nn.Sequential(
            nn.Conv3d(2,b,4,2,1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(b,b*2,4,2,1), nn.InstanceNorm3d(b*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(b*2,b*4,4,2,1), nn.InstanceNorm3d(b*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(b*4,1,1)
        )
    def forward(self,x,y): return self.net(torch.cat([x,y],1))


# ResNet3D 
class ResidualBlock3D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(ch, ch, 3, padding=1), nn.InstanceNorm3d(ch), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(ch, ch, 3, padding=1), nn.InstanceNorm3d(ch),
        )
        self.act = nn.LeakyReLU(0.1, inplace=True)
    def forward(self, x): return self.act(x + self.block(x))

class ResNet3D(nn.Module):
    """Residual 3D encoder-decoder (param-matched, base=16)."""
    def __init__(self, base=16):
        super().__init__(); b = base
        def enc(ic, oc): return nn.Sequential(nn.Conv3d(ic, oc, 3, padding=1), nn.InstanceNorm3d(oc), nn.LeakyReLU(0.1, inplace=True), ResidualBlock3D(oc))
        def dec(ic, oc): return nn.Sequential(nn.Conv3d(ic, oc, 3, padding=1), nn.InstanceNorm3d(oc), nn.LeakyReLU(0.1, inplace=True), ResidualBlock3D(oc))
        self.e1, self.e2, self.e3 = enc(1, b), enc(b, b*2), enc(b*2, b*4)
        self.bn = enc(b*4, b*8)
        self.d3, self.d2, self.d1 = dec(b*8+b*4, b*4), dec(b*4+b*2, b*2), dec(b*2+b, b)
        self.pool = nn.MaxPool3d(2); self.up = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=False)
        self.out = nn.Conv3d(b, 1, 1)
    def forward(self, x):
        s1=self.e1(x); s2=self.e2(self.pool(s1)); s3=self.e3(self.pool(s2)); btm=self.bn(self.pool(s3))
        x=self.d3(torch.cat([self.up(btm), s3], 1)); x=self.d2(torch.cat([self.up(x), s2], 1)); x=self.d1(torch.cat([self.up(x), s1], 1))
        return torch.sigmoid(self.out(x))

# DenseNet3D
class DenseLayer3D(nn.Module):
    def __init__(self, in_ch, growth=12):
        super().__init__()
        self.conv = nn.Sequential(nn.InstanceNorm3d(in_ch), nn.LeakyReLU(0.1, inplace=True), nn.Conv3d(in_ch, growth, 3, padding=1))
    def forward(self, x): return torch.cat([x, self.conv(x)], 1)

class DenseBlock3D(nn.Module):
    def __init__(self, in_ch, layers=3, growth=12):
        super().__init__(); self.layers = nn.ModuleList(); ch = in_ch
        for _ in range(layers): self.layers.append(DenseLayer3D(ch, growth)); ch += growth
        self.out_ch = ch
    def forward(self, x):
        for l in self.layers: x = l(x)
        return x

class DenseNet3D(nn.Module):
    """Dense-connected 3D encoder-decoder (param-matched, base=16, growth=12, 3 layers/block)."""
    def __init__(self, base=16, growth=12, nlayers=3):
        super().__init__(); b = base
        self.stem = nn.Conv3d(1, b, 3, padding=1)
        self.db1 = DenseBlock3D(b, nlayers, growth);    c1 = self.db1.out_ch
        self.tr1 = nn.Sequential(nn.Conv3d(c1, b*2, 1), nn.MaxPool3d(2))
        self.db2 = DenseBlock3D(b*2, nlayers, growth);  c2 = self.db2.out_ch
        self.tr2 = nn.Sequential(nn.Conv3d(c2, b*4, 1), nn.MaxPool3d(2))
        self.db3 = DenseBlock3D(b*4, nlayers, growth);  c3 = self.db3.out_ch
        self.tr3 = nn.Sequential(nn.Conv3d(c3, b*8, 1), nn.MaxPool3d(2))
        self.bn  = DenseBlock3D(b*8, nlayers, growth);  cbn = self.bn.out_ch
        self.up  = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=False)
        self.fuse3 = nn.Conv3d(cbn + c3, b*4, 1); self.udb3 = DenseBlock3D(b*4, nlayers, growth); cu3 = self.udb3.out_ch
        self.fuse2 = nn.Conv3d(cu3 + c2, b*2, 1); self.udb2 = DenseBlock3D(b*2, nlayers, growth); cu2 = self.udb2.out_ch
        self.fuse1 = nn.Conv3d(cu2 + c1, b,   1); self.udb1 = DenseBlock3D(b,   nlayers, growth); cu1 = self.udb1.out_ch
        self.out = nn.Conv3d(cu1, 1, 1)
    def forward(self, x):
        x = self.stem(x)
        s1=self.db1(x);  s2=self.db2(self.tr1(s1)); s3=self.db3(self.tr2(s2)); btm=self.bn(self.tr3(s3))
        x=self.udb3(self.fuse3(torch.cat([self.up(btm), s3], 1)))
        x=self.udb2(self.fuse2(torch.cat([self.up(x),   s2], 1)))
        x=self.udb1(self.fuse1(torch.cat([self.up(x),   s1], 1)))
        return torch.sigmoid(self.out(x))

# Nearest-Neighbour (non-parametric) 
class NearestNeighborBaseline:
    """Retrieve the training PET whose MRI is closest (L2 on 16³ thumbnail)."""
    def __init__(self): self.keys = []; self.vals = []
    def fit(self, loader, max_samples=200):
        import skimage.transform as T
        print('Building NN index …'); n = 0
        for mri, pet in loader:
            mri_np = mri.squeeze().numpy(); pet_np = pet.squeeze().numpy()
            thumb = T.resize(mri_np, (16,16,16), anti_aliasing=True).ravel().astype(np.float32)
            self.keys.append(thumb); self.vals.append(pet_np.astype(np.float32))
            n += 1
            if n >= max_samples: break
        self.keys = np.stack(self.keys)
        print(f'  indexed {n} training volumes')
    def predict(self, mri_np):
        import skimage.transform as T
        q = T.resize(mri_np, (16,16,16), anti_aliasing=True).ravel().astype(np.float32)
        dists = np.sum((self.keys - q)**2, axis=1)
        return self.vals[int(np.argmin(dists))]
    # make eval_model-compatible by mimicking .eval() and keeping state on CPU
    def eval(self): return self
    def __call__(self, x): raise NotImplementedError("use predict() directly")

print("ResNet3D, DenseNet3D, NearestNeighborBaseline defined.")


ResNet3D, DenseNet3D, NearestNeighborBaseline defined.


## Losses and sliding-window inference

In [7]:
def gaussian_kernel_3d(size=7, sigma=1.5, device='cpu'):
    coords = torch.arange(size, dtype=torch.float32, device=device) - size//2
    g1d = torch.exp(-coords**2/(2*sigma**2)); g1d = g1d/g1d.sum()
    kernel = g1d[:,None,None]*g1d[None,:,None]*g1d[None,None,:]
    return kernel.unsqueeze(0).unsqueeze(0)
_KERNEL=None

def ssim_loss(pred, target, kernel_size=7, sigma=1.5, C1=1e-4, C2=9e-4):
    global _KERNEL
    if _KERNEL is None or _KERNEL.device!=pred.device:
        _KERNEL = gaussian_kernel_3d(kernel_size, sigma, pred.device)
    pad=kernel_size//2
    mu1=F.conv3d(pred,_KERNEL,padding=pad); mu2=F.conv3d(target,_KERNEL,padding=pad)
    mu1sq,mu2sq,mu12 = mu1**2, mu2**2, mu1*mu2
    sig1sq = F.conv3d(pred*pred,_KERNEL,padding=pad)-mu1sq
    sig2sq = F.conv3d(target*target,_KERNEL,padding=pad)-mu2sq
    sig12  = F.conv3d(pred*target,_KERNEL,padding=pad)-mu12
    num=(2*mu12+C1)*(2*sig12+C2); den=(mu1sq+mu2sq+C1)*(sig1sq+sig2sq+C2)
    return 1.0 - (num/den).mean()

def gdl_loss(pred, target, alpha=2):
    def grad(v, dim): return v.narrow(dim,1,v.shape[dim]-1) - v.narrow(dim,0,v.shape[dim]-1)
    loss=0.0
    for dim in (2,3,4):
        gp,gt = grad(pred,dim), grad(target,dim)
        loss += (torch.abs(torch.abs(gp)-torch.abs(gt))**alpha).mean()
    return loss/3.0

def combined_loss(pred, target, lam_l1=0.15, lam_ssim=0.55, lam_gdl=0.30):
    l1 = F.l1_loss(pred, target); ssim = ssim_loss(pred, target); gdl = gdl_loss(pred, target)
    return lam_l1*l1 + lam_ssim*ssim + lam_gdl*gdl

def _single_pass_infer(model, mri_np, patch=PATCH_SIZE, overlap=0.5):
    model.eval(); D,H,W = mri_np.shape; step=max(1,int(patch*(1-overlap)))
    win1d = np.hanning(patch).astype(np.float32)+1e-6; win3d = win1d[:,None,None]*win1d[None,:,None]*win1d[None,None,:]
    pad_d=(patch-D%patch)%patch; pad_h=(patch-H%patch)%patch; pad_w=(patch-W%patch)%patch
    mri_p=np.pad(mri_np, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect'); PD,PH,PW=mri_p.shape
    out=np.zeros((PD,PH,PW),dtype=np.float32); wgt=np.zeros_like(out)
    xs = list(range(0, PD-patch+1, step)) or [0]
    ys = list(range(0, PH-patch+1, step)) or [0]
    zs = list(range(0, PW-patch+1, step)) or [0]

    if xs[-1] + patch < PD:
        xs.append(PD - patch)
    
    if ys[-1] + patch < PH:
        ys.append(PH - patch)
    
    if zs[-1] + patch < PW:
        zs.append(PW - patch)
    with torch.no_grad():
        for x in xs:
            for y in ys:
                for z in zs:
                    t=torch.from_numpy(mri_p[x:x+patch,y:y+patch,z:z+patch]).unsqueeze(0).unsqueeze(0).to(DEVICE)
                    pred=model(t)[0,0].cpu().numpy(); out[x:x+patch,y:y+patch,z:z+patch]+=pred*win3d; wgt[x:x+patch,y:y+patch,z:z+patch]+=win3d
    return (out/(wgt+1e-8))[:D,:H,:W].astype(np.float32)

def sliding_window_infer(model, mri_np, patch=PATCH_SIZE, overlap=0.5, tta=False):
    if not tta: return _single_pass_infer(model, mri_np, patch, overlap)
    acc=np.zeros_like(mri_np,dtype=np.float32); flips=[[],[0],[1],[2],[0,1],[0,2],[1,2],[0,1,2]]
    for axes in flips:
        aug=mri_np.copy();
        for ax in axes: aug=np.flip(aug,axis=ax)
        pred=_single_pass_infer(model, np.ascontiguousarray(aug), patch, overlap)
        for ax in axes: pred=np.flip(pred,axis=ax)
        acc+=np.ascontiguousarray(pred)
    return (acc/len(flips)).astype(np.float32)


## Training loop (shared hyperparams)

In [8]:
def train_model(model, loss_fn, epochs=EPOCHS, adv_parts=None, ckpt_name=None):
    ckpt_path = os.path.join(CKPT_DIR, f"{ckpt_name}.pt") if ckpt_name else None
    start_ep  = 1

    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scaler = torch.amp.GradScaler('cuda') if DEVICE == 'cuda' else None
    bce    = nn.BCEWithLogitsLoss()

    # ── Resume if checkpoint exists ───────────────────────────────────────
    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model'])
        opt.load_state_dict(ckpt['opt'])
        start_ep = ckpt['epoch'] + 1
        print(f"  [resume] {ckpt_name} from epoch {ckpt['epoch']}/{epochs} "
              f"(loss={ckpt.get('loss', float('nan')):.4f})")
        if start_ep > epochs:
            print(f"  [resume] already complete — skipping training.")
            return model

    for ep in range(start_ep, epochs + 1):
        model.train(); tl = []
        for mri, pet in train_loader:
            mri, pet = mri.to(DEVICE), pet.to(DEVICE)
            if adv_parts:
                G = model; D = adv_parts['D']
                optD = adv_parts['optD']; lambda_l1 = adv_parts['lambda_l1']
                # Train D
                with torch.no_grad(): fake = G(mri)
                optD.zero_grad(set_to_none=True)
                pred_real = D(mri, pet); pred_fake = D(mri, fake.detach())
                loss_D = 0.5 * (bce(pred_real, torch.ones_like(pred_real)) +
                                bce(pred_fake, torch.zeros_like(pred_fake)))
                loss_D.backward(); optD.step()
                # Train G
                opt.zero_grad(set_to_none=True); fake = G(mri); pred_fake = D(mri, fake)
                loss = bce(pred_fake, torch.ones_like(pred_fake)) + lambda_l1 * F.l1_loss(fake, pet)
            else:
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type='cuda', enabled=DEVICE == 'cuda'):
                    loss = loss_fn(model(mri), pet)
            if scaler:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            tl.append(loss.item())

        ep_loss = float(np.mean(tl))
        if ep % 5 == 0:
            print(f"Epoch {ep}/{epochs} | train loss {ep_loss:.4f}")

        # Save checkpoint every 10 epochs and at the final epoch
        if ckpt_path and (ep % 10 == 0 or ep == epochs):
            torch.save({'epoch': ep, 'loss': ep_loss,
                        'model': model.state_dict(),
                        'opt':   opt.state_dict()}, ckpt_path)

    return model


## Train the four models

In [9]:
loss_proposed = lambda pred, tgt: combined_loss(pred, tgt, lam_l1=0.15, lam_ssim=0.55, lam_gdl=0.30)
loss_l1       = lambda pred, tgt: F.l1_loss(pred, tgt)

print('\nTraining Proposed v3 (Attention UNet + triple loss)')
proposed   = UNetAtt(base=16)
proposed   = train_model(proposed,   loss_proposed, ckpt_name='proposed_v3')

print('\nTraining UNet-L1')
unet_l1    = UNetPlain(base=16)
unet_l1    = train_model(unet_l1,    loss_l1,       ckpt_name='unet_l1')

print('\nTraining UNet-SSIM+GDL (no attention)')
unet_tl    = UNetPlain(base=16)
unet_tl    = train_model(unet_tl,    loss_proposed, ckpt_name='unet_ssim_gdl')

print('\nTraining Pix2Pix3D (GAN)')
pixG  = UNetPlain(base=16)
pixD  = PatchDiscriminator3D(base=16).to(DEVICE)
optD  = torch.optim.AdamW(pixD.parameters(), lr=LR, weight_decay=WD)
adv_cfg = {'D': pixD, 'optD': optD, 'lambda_l1': 100.0}
pixG  = train_model(pixG,  loss_l1, adv_parts=adv_cfg, ckpt_name='pix2pix3d')

print('\nTraining ResNet3D')
resnet3d   = ResNet3D(base=16)
resnet3d   = train_model(resnet3d,   loss_proposed, ckpt_name='resnet3d')

print('\nTraining DenseNet3D')
densenet3d = DenseNet3D(base=16)
densenet3d = train_model(densenet3d, loss_proposed, ckpt_name='densenet3d')

print('\nBuilding Nearest-Neighbour index (no training)')
nn_baseline = NearestNeighborBaseline()
nn_baseline.fit(train_loader, max_samples=200)



Training Proposed v3 (Attention UNet + triple loss)
Epoch 5/20 | train loss 0.2252
Epoch 10/20 | train loss 0.2215
Epoch 15/20 | train loss 0.2104
Epoch 20/20 | train loss 0.2206

Training UNet-L1
Epoch 5/20 | train loss 0.1562
Epoch 10/20 | train loss 0.1467
Epoch 15/20 | train loss 0.1436
Epoch 20/20 | train loss 0.1412

Training UNet-SSIM+GDL (no attention)
Epoch 5/20 | train loss 0.2556
Epoch 10/20 | train loss 0.2371
Epoch 15/20 | train loss 0.2276
Epoch 20/20 | train loss 0.2250

Training Pix2Pix3D (GAN)
Epoch 5/20 | train loss 16.3626
Epoch 10/20 | train loss 16.1652
Epoch 15/20 | train loss 15.8751
Epoch 20/20 | train loss 15.7254

Training ResNet3D
Epoch 5/20 | train loss 0.2347
Epoch 10/20 | train loss 0.2263
Epoch 15/20 | train loss 0.2180
Epoch 20/20 | train loss 0.2180

Training DenseNet3D
Epoch 5/20 | train loss 0.2223
Epoch 10/20 | train loss 0.2322
Epoch 15/20 | train loss 0.3071
Epoch 20/20 | train loss 0.4345

Building Nearest-Neighbour index (no training)
Building N

## Save / Resume Models

In [10]:
MODEL_REGISTRY = {
    'proposed_v3':  (UNetAtt,   16, proposed),
    'unet_l1':      (UNetPlain, 16, unet_l1),
    'unet_ssim_gdl':(UNetPlain, 16, unet_tl),
    'pix2pix3d':    (UNetPlain, 16, pixG),
    'resnet3d':     (ResNet3D,  16, resnet3d),
    'densenet3d':   (DenseNet3D,16, densenet3d),
}

def save_all_models(registry=MODEL_REGISTRY):
    for name, (cls, base, mdl) in registry.items():
        path = os.path.join(CKPT_DIR, f"{name}_final.pt")
        torch.save({'arch': cls.__name__, 'base': base,
                    'model': mdl.state_dict()}, path)
        print(f"  saved {path}")

def load_all_models():
    """ Reload all models from *_final.pt files without re-training."""
    arch_map = {'UNetAtt': UNetAtt, 'UNetPlain': UNetPlain,
                'ResNet3D': ResNet3D, 'DenseNet3D': DenseNet3D}
    loaded = {}
    for name, (cls, base, _) in MODEL_REGISTRY.items():
        path = os.path.join(CKPT_DIR, f"{name}_final.pt")
        if not os.path.exists(path):
            print(f"  [warn] {path} not found -- skipping"); continue
        ckpt = torch.load(path, map_location=DEVICE)
        mdl  = arch_map[ckpt['arch']](base=ckpt['base']).to(DEVICE)
        mdl.load_state_dict(ckpt['model']); mdl.eval()
        loaded[name] = mdl
        print(f"  loaded {path}")
    return loaded

save_all_models()
print(f"\nAll final weights saved to {CKPT_DIR}")
print("Call load_all_models() at the start of a new session to skip retraining.")


  saved /workspace/checkpoints/benchmark_v3/proposed_v3_final.pt
  saved /workspace/checkpoints/benchmark_v3/unet_l1_final.pt
  saved /workspace/checkpoints/benchmark_v3/unet_ssim_gdl_final.pt
  saved /workspace/checkpoints/benchmark_v3/pix2pix3d_final.pt
  saved /workspace/checkpoints/benchmark_v3/resnet3d_final.pt
  saved /workspace/checkpoints/benchmark_v3/densenet3d_final.pt

All final weights saved to /workspace/checkpoints/benchmark_v3
Call load_all_models() at the start of a new session to skip retraining.


## Evaluation (brain-masked metrics)

In [11]:
@torch.no_grad()
def eval_model(model, name):
    model.eval(); rows=[]
    for i,(mri,pet) in enumerate(test_loader):
        mri_np = mri.squeeze(0).squeeze(0).numpy(); pet_np = pet.squeeze(0).squeeze(0).numpy(); mask=compute_brain_mask(mri_np)
        syn_np = sliding_window_infer(model, mri_np, tta=USE_TTA)
        m = compute_masked_metrics(syn_np, pet_np, mask); m['model']=name; m['subject']=f's{i:02d}'; rows.append(m)
    return pd.DataFrame(rows)

df_proposed = eval_model(proposed, 'Proposed v3')
df_unet_l1  = eval_model(unet_l1, 'UNet-L1')
df_unet_tl  = eval_model(unet_tl, 'UNet-SSIM+GDL')
df_pix      = eval_model(pixG, 'Pix2Pix3D')
df_resnet   = eval_model(resnet3d,   'ResNet3D')
df_densenet = eval_model(densenet3d, 'DenseNet3D')

# Nearest-neighbour: rebuild index with the fix, then eval
class NearestNeighborBaseline:
    def __init__(self): self.keys = []; self.vals = []
    def fit(self, loader, max_samples=200):
        import skimage.transform as T
        print('Building NN index …'); n = 0
        for mri, pet in loader:
            mri_np = mri[0, 0].numpy()   # fix: explicit indexing avoids batch-size squeeze bug
            pet_np = pet[0, 0].numpy()
            thumb = T.resize(mri_np, (16,16,16), anti_aliasing=True).ravel().astype(np.float32)
            self.keys.append(thumb); self.vals.append(pet_np.astype(np.float32))
            n += 1
            if n >= max_samples: break
        self.keys = np.stack(self.keys)  # shape (N, 4096)
        print(f'  indexed {n} training volumes (key dim={self.keys.shape[1]})')
    def predict(self, mri_np):
        import skimage.transform as T
        q = T.resize(mri_np, (16,16,16), anti_aliasing=True).ravel().astype(np.float32)
        dists = np.sum((self.keys - q)**2, axis=1)
        return self.vals[int(np.argmin(dists))]
    def eval(self): return self

nn_baseline = NearestNeighborBaseline()
nn_baseline.fit(train_loader, max_samples=200)

nn_rows = []
for i, (mri, pet) in enumerate(test_loader):
    mri_np = mri.squeeze(0).squeeze(0).numpy()
    pet_np = pet.squeeze(0).squeeze(0).numpy()
    mask   = compute_brain_mask(mri_np)
    syn_np = nn_baseline.predict(mri_np)
    if syn_np.shape != pet_np.shape:
        import skimage.transform as T
        syn_np = T.resize(syn_np, pet_np.shape, anti_aliasing=True).astype(np.float32)
    m = compute_masked_metrics(syn_np, pet_np, mask)
    m['model'] = 'NearestNeighbor'; m['subject'] = f's{i:02d}'; nn_rows.append(m)
df_nn = pd.DataFrame(nn_rows)

df_all = pd.concat([df_proposed, df_unet_l1, df_unet_tl, df_pix,
                    df_resnet, df_densenet, df_nn], ignore_index=True)
print("All models evaluated. Shape:", df_all.shape)
df_all.head()

Building NN index …
  indexed 200 training volumes (key dim=4096)
All models evaluated. Shape: (1701, 8)


,MAE,MSE,SSIM,MAE_brain,MSE_brain,SSIM_brain,model,subject
0,0.097567,0.017130,0.605818,0.108185,0.016510,0.672360,Proposed v3,s00
1,0.130183,0.029066,0.470875,0.115772,0.020752,0.566607,Proposed v3,s01
2,0.139111,0.048836,0.478081,0.179067,0.062062,0.572339,Proposed v3,s02
3,0.204956,0.082726,0.384837,0.205798,0.080313,0.455315,Proposed v3,s03
4,0.144414,0.043172,0.404785,0.161493,0.049169,0.461201,Proposed v3,s04


## Plots

In [12]:
sns.set_style('whitegrid')
palette = {
    'Proposed v3':   '#1f78b4',
    'UNet-L1':       '#33a02c',
    'UNet-SSIM+GDL': '#6a3d9a',
    'Pix2Pix3D':     '#e31a1c',
    'ResNet3D':       '#ff7f00',
    'DenseNet3D':     '#b15928',
    'NearestNeighbor':'#999999',
}
MODEL_ORDER = list(palette.keys())
METRICS = [
    ('SSIM_brain', 'Brain SSIM ↑',  True),
    ('MAE_brain',  'Brain MAE ↓',   False),
    ('MSE_brain',  'Brain MSE ↓',   False),
]

# ── 1. Bar chart with SE error bars (one subplot per metric) ─────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (metric, label, higher_better) in zip(axes, METRICS):
    sns.barplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,
                estimator=np.mean, errorbar='se', palette=palette, ax=ax)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel(label)
    ax.tick_params(axis='x', rotation=30)
    # Annotate best bar
    means = df_all.groupby('model')[metric].mean()
    best  = means.idxmax() if higher_better else means.idxmin()
    best_pos = MODEL_ORDER.index(best)
    ax.get_children()[best_pos].set_edgecolor('gold')
    ax.get_children()[best_pos].set_linewidth(2.5)
plt.suptitle('Model Comparison — Mean ± SE (brain-masked)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('benchmark_bars.png', dpi=300, bbox_inches='tight')
plt.show(); print('Saved benchmark_bars.png')

# ── 2. Strip + box per metric (per-subject spread) ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (metric, label, _) in zip(axes, METRICS):
    sns.boxplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,
                palette=palette, boxprops={'alpha': 0.35}, ax=ax,
                fliersize=0, width=0.55)
    sns.stripplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,
                  palette=palette, jitter=True, size=5, alpha=0.7, ax=ax)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel(label)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Per-subject Distribution (brain-masked)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('benchmark_distribution.png', dpi=300, bbox_inches='tight')
plt.show(); print('Saved benchmark_distribution.png')

# ── 3. Per-subject line chart — SSIM_brain across subjects ───────────────
fig, ax = plt.subplots(figsize=(12, 4.5))
for model_name, grp in df_all.groupby('model'):
    grp_sorted = grp.sort_values('subject')
    ax.plot(grp_sorted['subject'], grp_sorted['SSIM_brain'],
            marker='o', markersize=4, label=model_name,
            color=palette.get(model_name, None), alpha=0.85)
ax.set_title('Per-subject Brain SSIM — all models', fontsize=12, fontweight='bold')
ax.set_xlabel('Subject'); ax.set_ylabel('SSIM (brain-masked)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('benchmark_per_subject_ssim.png', dpi=300, bbox_inches='tight')
plt.show(); print('Saved benchmark_per_subject_ssim.png')

# ── 4. Ranked summary heatmap ─────────────────────────────────────────────
agg = df_all.groupby('model')[
    [m for m, _, _ in METRICS]
].mean().reindex(MODEL_ORDER)

# Normalise each column to [0, 1] (1 = best)
norm_agg = agg.copy()
for metric, _, higher_better in METRICS:
    col = agg[metric]
    rng = col.max() - col.min() + 1e-8
    norm_agg[metric] = (col - col.min()) / rng if higher_better else (col.max() - col) / rng

norm_agg.columns = [label for _, label, _ in METRICS]
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(norm_agg, annot=agg.round(4).values, fmt='', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Normalised performance (green = best)', fontsize=11, fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
plt.savefig('benchmark_heatmap.png', dpi=300, bbox_inches='tight')
plt.show(); print('Saved benchmark_heatmap.png')


/tmp/ipykernel_37924/3035646240.py:21: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,
/tmp/ipykernel_37924/3035646240.py:21: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,
/tmp/ipykernel_37924/3035646240.py:21: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,


Saved benchmark_bars.png


/tmp/ipykernel_37924/3035646240.py:40: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,
/tmp/ipykernel_37924/3035646240.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,
/tmp/ipykernel_37924/3035646240.py:40: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_all, x='model', y=metric, order=MODEL_ORDER,
/tmp/ipykernel_37924/3035646240.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed

Saved benchmark_distribution.png
Saved benchmark_per_subject_ssim.png
Saved benchmark_heatmap.png


## Qualitative mid-slice grid

In [13]:
@torch.no_grad()
def save_qualitative(idx=0, out='qualitative_mid_slice.png'):
    import skimage.transform as T
    mri_path, pet_path = te_pairs[idx]
    mri_np = norm_mri(np.load(mri_path)); pet_np = norm_pet(np.load(pet_path)); mid=mri_np.shape[0]//2
    
    # NN returns a stored 64³ patch — resize to full volume shape first
    nn_pred = nn_baseline.predict(mri_np)
    if nn_pred.shape != mri_np.shape:
        nn_pred = T.resize(nn_pred, mri_np.shape, anti_aliasing=True).astype(np.float32)

    preds={
        'Proposed v3':   sliding_window_infer(proposed,   mri_np, tta=USE_TTA),
        'UNet-L1':       sliding_window_infer(unet_l1,    mri_np, tta=USE_TTA),
        'UNet-SSIM+GDL': sliding_window_infer(unet_tl,    mri_np, tta=USE_TTA),
        'Pix2Pix3D':     sliding_window_infer(pixG,       mri_np, tta=USE_TTA),
        'ResNet3D':      sliding_window_infer(resnet3d,   mri_np, tta=USE_TTA),
        'DenseNet3D':    sliding_window_infer(densenet3d, mri_np, tta=USE_TTA),
        'NearestNeighbor': nn_pred,
    }
    imgs=[mri_np[mid], pet_np[mid]] + [preds[k][mid] for k in preds]
    titles=['MRI','Real PET'] + list(preds.keys()); cmaps=['gray','hot'] + ['hot']*len(preds)
    fig,axes=plt.subplots(1,len(imgs), figsize=(24,3))
    for ax,im,t,cm in zip(axes, imgs, titles, cmaps):
        ax.imshow(im, cmap=cm, vmin=0, vmax=1); ax.set_title(t, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.savefig(out, dpi=300); plt.close(fig); print('Saved', out)

save_qualitative(0, 'qualitative_mid_slice.png')

Saved qualitative_mid_slice.png


## Numeric summary CSVs

In [14]:
summary = df_all.groupby('model')[['SSIM_brain','MAE_brain','MSE_brain']].agg(['mean','std'])
print(summary)
summary.to_csv('benchmark_metrics_summary.csv')
df_all.to_csv('benchmark_per_subject.csv', index=False)
print('Saved benchmark_metrics_summary.csv and benchmark_per_subject.csv')


                SSIM_brain           MAE_brain           MSE_brain          
                      mean       std      mean       std      mean       std
model                                                                       
DenseNet3D        0.248577  0.275458  0.517885  0.176822  0.359067  0.150921
NearestNeighbor   0.354003  0.172234  0.322075  0.127291  0.162577  0.120321
Pix2Pix3D         0.588959  0.180614  0.174731  0.096295  0.056175  0.069629
Proposed v3       0.624434  0.202898  0.172947  0.112005  0.055101  0.076521
ResNet3D          0.622227  0.194017  0.179543  0.108521  0.058390  0.072754
UNet-L1           0.606909  0.182505  0.175641  0.102121  0.055123  0.068745
UNet-SSIM+GDL     0.610376  0.184906  0.199941  0.126161  0.072626  0.095163
Saved benchmark_metrics_summary.csv and benchmark_per_subject.csv
